![Ticket 2 Ride Banner](images/banner.png)

# Ticket 2 ride by Ramine and Fatima

## Explore and check the data first

In [ ]:
import pandas as pd

df = pd.read_csv("data/passagierfrequenz.csv", sep=";")

df.head()

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df["Jahr_Annee_Anno"].unique()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.duplicated(
    subset=["UIC", "Jahr_Annee_Anno"]
).sum()

In [ ]:
df_2025 = df[df["Jahr_Annee_Anno"] == 2025]

df_2025.shape

In [ ]:
top10 = df_2025.sort_values(
    by="DTV_TJM_TGM",
    ascending=False
).head(10)

top10[["Bahnhof_Gare_Stazione", "DTV_TJM_TGM"]]

## 1. Find the busiest stations

In [ ]:
import matplotlib.pyplot as plt
plt.barh(
    top10["Bahnhof_Gare_Stazione"],
    top10["DTV_TJM_TGM"]
)
plt.xlabel("Average daily passengers") 
plt.title("top 10 busiest swiss stations (2025)")
plt.gca().invert_yaxis()

plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.barh(
    top10["Bahnhof_Gare_Stazione"],
    top10["DTV_TJM_TGM"]
)
plt.xlabel("Average daily passengers") 
plt.title("top 10 busiest swiss stations (2025)")
plt.gca().invert_yaxis()

#add our source 
plt.figtext(
    0.5, -0.02,
    "source: SBB open data (PF)",
    ha="center",
    fontsize=9
) 
plt.show()


## 2. Cornavin's story

In [ ]:
geneva = df[df["Bahnhof_Gare_Stazione"]=="Genève"]
geneva = geneva.sort_values("Jahr_Annee_Anno") 
geneva[["Jahr_Annee_Anno","DTV_TJM_TGM"]]

In [ ]:
import matplotlib.pyplot as plt

plt.plot(
    geneva["Jahr_Annee_Anno"],
    geneva["DTV_TJM_TGM"],
    marker="o"
)

plt.title("Cornavins story (2018 to 2025)")
plt.xlabel("Year") 
plt.ylabel("Average daily passengers")
plt.xticks(geneva["Jahr_Annee_Anno"])
plt.grid(True)
plt.show()

In [ ]:
geneva["Growth%"
    ] = geneva["DTV_TJM_TGM"].pct_change() * 100
geneva[["Jahr_Annee_Anno", "DTV_TJM_TGM", "Growth%"]]

### Findings 
Passenger traffic at Cornavin increased from 60k average daily passengers in 2018 to 83k in 2025, that is a huge growth and it's almost 37,4% average increase. Also, the largest increase from one year to another happened in the year 23-23 (with +18.83%). Growth then slowed to 1.38% in 2024 and then reached 3.10% in 2025.
We couldn't analyse 2019, 2020, 2021 because the dataset didnt include them (they're missing). 

## 3.The Swiss Rail Map

In [ ]:
# on charge pandas (pd) pour les tableaux et matplotlib (plt) pour les graphiques
import pandas as pd
import matplotlib.pyplot as plt
# on lit le fichier passagierfrequenz.csv (ses colonnes sont séparées par des ;)
df = pd.read_csv("data/passagierfrequenz.csv", sep=";")


In [ ]:
# on garde seulement les lignes de l'année 2025
df_2025 = df[df["Jahr_Annee_Anno"] == 2025]
df_2025.shape

In [ ]:
# combien de gares n'ont pas de coordonnées ?
df_2025["geopos"].isna().sum()

In [ ]:
# on enlève les gares qui n'ont pas de coordonnées en regardant la colonne geopos
df_2025 = df_2025.dropna(subset=["geopos"])
df_2025.shape

In [ ]:
# on coupe geopos à la virgule pour toutes les gares et après on met latitude et longitude dans deux colonnes (0 et 1)
coord = df_2025["geopos"].str.split(",", expand=True)
coord.head()

In [ ]:
# on transforme les textes en nombres et on crée les colonnes lat et lon. Ensuite vérification.
df_2025["lat"] = coord[0].astype(float)
df_2025["lon"] = coord[1].astype(float)
df_2025[["Bahnhof_Gare_Stazione", "lat", "lon"]].head()

In [ ]:
# une feuille de dessin plus grande (largeur 9, hauteur 6)
plt.figure(figsize=(9, 6))

# fond : toutes les gares en petits points gris, pour voir le réseau
plt.scatter(df_2025["lon"], df_2025["lat"], s=2, color="lightgray")
# par-dessus : taille du point = montées et descentes par jour, divisées par 1000
# alpha=0.5 rend les bulles transparentes, pour voir celles qui se chevauchent
plt.scatter(df_2025["lon"], df_2025["lat"], s=df_2025["DTV_TJM_TGM"] / 1000, color="tab:blue", alpha=0.5)

# légende des tailles, avec 3 bulles d'exemple qui ne sont pas placées sur la carte
plt.scatter([], [], s=10000 / 1000, color="tab:blue", alpha=0.5, label="10,000")
plt.scatter([], [], s=100000 / 1000, color="tab:blue", alpha=0.5, label="100,000")
plt.scatter([], [], s=400000 / 1000, color="tab:blue", alpha=0.5, label="400,000")
# légende sans cadre, avec un texte plus petit
plt.legend(title="Boardings + alightings per day", loc="upper left", fontsize=8, title_fontsize=10, labelspacing=1.8, frameon=False)

# vraies proportions de la Suisse (1 degré nord-sud est 1,46 fois plus long qu'1 degré est-ouest)
plt.gca().set_aspect(1.46)

# titre et noms des axes
plt.title("Busiest Swiss train stations (2025)")
plt.xlabel("Longitude (degrees)")
plt.ylabel("Latitude (degrees)")
# la source sous la carte
plt.figtext(0.5, -0.02, "source: SBB open data (PF)", ha="center", fontsize=9)
plt.show()

### What we found
Les plus grosses bulles se trouvent surtout en Suisse alémanique, autour de Zurich, et huit des dix gares les plus fréquentées sont alémaniques. Zürich HB est très loin devant, avec 449 800 montées et descentes par jour, environ 5 fois plus que Genève (83 100). La Suisse romande a moins de grosses bulles, même si Lausanne et Genève sont dans le top 10, et au Tessin et dans les Alpes on ne voit presque que de petites gares.

Attention, on compte des montées et des descentes, pas des voyageurs. Une personne qui change de train à Zurich est comptée deux fois, ce qui grossit les grandes gares de correspondance.

## 4.Your station, please!

### intro
La fonction analyser_gare(nom) affiche les chiffres d'une gare choisie : les années disponibles, la fréquentation de la dernière année, l'évolution depuis la première année et les remarques des CFF, avec un graphique de 2018 à 2025. Il suffit d'écrire par exemple analyser_gare("Genève"). Le nom doit être écrit exactement comme dans le fichier, avec les accents, sinon la fonction affiche un message. Attention, on compte des montées et des descentes, pas des voyageurs, et la valeur 49 veut dire moins de 50.

In [ ]:
# on charge pandas (pd) pour les tableaux et matplotlib (plt) pour les graphiques
import pandas as pd
import matplotlib.pyplot as plt
# on lit le fichier passagierfrequenz.csv (ses colonnes sont séparées par des ;)
df = pd.read_csv("data/passagierfrequenz.csv", sep=";")

In [ ]:
def analyser_gare(nom):
    # on enlève les espaces au début et à la fin du nom tapé
    nom = nom.strip()
    # si on n'a rien tapé, on s'arrête tout de suite
    if nom == "":
        print("Écris un nom de gare.")
        return

    # 1. on garde les lignes de la gare demandée, triées par année
    gare = df[df["Bahnhof_Gare_Stazione"] == nom]
    gare = gare.sort_values("Jahr_Annee_Anno")
    annees = gare["Jahr_Annee_Anno"].tolist()
    valeurs = gare["DTV_TJM_TGM"].tolist()

    # si la liste est vide, le nom n'existe pas : on propose des gares proches et on s'arrête
    if len(annees) == 0:
        print(f"Gare introuvable : {nom}")
        noms = df["Bahnhof_Gare_Stazione"]
        proches = noms[noms.str.contains(nom, case=False, regex=False)].unique().tolist()
        if len(proches) > 0:
            print("Tu voulais peut-être dire :", proches)
        else:
            print("Vérifie l'orthographe exacte, avec les accents (par exemple Zürich HB ou Genève).")
        return

    # 2. on affiche les chiffres
    print("Gare :", nom)
    print("Années disponibles :", annees)
    print(f"En {annees[-1]} : {int(valeurs[-1])} montées et descentes par jour")
    # l'évolution n'a de sens que s'il y a au moins deux années
    if len(annees) > 1:
        evolution = (valeurs[-1] - valeurs[0]) / valeurs[0] * 100
        print(f"Évolution depuis {annees[0]} : {evolution:.1f} %")

    # les remarques des CFF sur cette gare (changements de comptage, trains de l'étranger...)
    remarques = gare["Remarques"].dropna().unique().tolist()
    if len(remarques) > 0:
        print("Remarques des CFF :", remarques)

    # 3. on dessine le graphique, avec toujours les années 2018 à 2025
    plt.bar(annees, valeurs)
    plt.xticks(range(2018, 2026))
    plt.title(f"{nom}: boardings + alightings per day")
    plt.xlabel("Year")
    plt.ylabel("Boardings + alightings per day")
    plt.figtext(0.5, -0.02, "source: SBB open data (PF)", ha="center", fontsize=9)
    plt.xticks(gare["Jahr_Annee_Anno"])
    plt.show()

In [ ]:
analyser_gare("Genève")

In [ ]:
analyser_gare("Lausanne")

In [ ]:
analyser_gare("Lancy-Bachet")

In [ ]:
analyser_gare("Zurich HBB")

In [ ]:
analyser_gare("Zürich HB")

### Antisèche : noms exacts des gares pour la démo

Le nom doit être écrit exactement comme dans le fichier des CFF. Si on se trompe, la fonction propose les gares qui contiennent le mot tapé.

| On pense à… | On écrit |
|---|---|
| Zurich, gare centrale | Zürich HB |
| Aéroport de Zurich | Zürich Flughafen |
| Genève Cornavin | Genève |
| Aéroport de Genève | Genève-Aéroport |
| Bâle | Basel SBB |
| Berne | Bern |
| Lucerne | Luzern |
| Fribourg | Fribourg/Freiburg |
| Bienne | Biel/Bienne |
| Sierre | Sierre/Siders |
| Renens | Renens VD |
| Saint-Gall | St. Gallen |
| Saint-Moritz | St. Moritz |
| Coire | Chur |
| Interlaken | Interlaken Ost |
| Yverdon | Yverdon-les-Bains |

Noms simples, qui marchent tels quels : Lausanne, Winterthur, Olten, Zug, Thun, Aarau, Baden, Nyon, Morges, Vevey, Montreux, Neuchâtel, Sion, Martigny, Delémont, Lugano, Bellinzona, Locarno, Brig, Visp, Zermatt, Lancy-Pont-Rouge, Lancy-Bachet.

In [ ]:
gare_nom = input("Veuillez écrire un nom d'une gare ? :")
analyser_gare(gare_nom)

## 5. Passenger Power

In [ ]:
# on charge pandas pour les tableaux et matplotlib pour les graphiques
import pandas as pd
import matplotlib.pyplot as plt

# on lit le fichier des CFF et les colonnes sont séparé par ;
df = pd.read_csv("data/passagierfrequenz.csv", sep=";")

In [ ]:
# les 5 grandes gares qu'on va comparer
gares = ["Zürich HB", "Bern", "Lausanne", "Genève", "Luzern"]

# on garde seulement l'année 2025
df_2025 = df[df["Jahr_Annee_Anno"] == 2025]

# on garde seulement les lignes des 5 gares de la liste
cinq = df_2025[df_2025["Bahnhof_Gare_Stazione"].isin(gares)]

# on garde 3 colonnes : le nom, le numéro UIC et les montées + descentes par jour
cinq = cinq[["Bahnhof_Gare_Stazione", "UIC", "DTV_TJM_TGM"]].copy()

# on transforme 8501008.0 en "8501008" pour pouvoir le comparer avec les horaires
cinq["didok"] = cinq["UIC"].astype(int).astype(str)
cinq

In [ ]:
# le dossier où on a dézippé les horaires
dossier = "data/gtfs_fp2026_20261118/"

# on lit la liste de tous les arrêts de Suisse, tout en texte (dtype=str)
stops = pd.read_csv(dossier + "stops.txt", dtype=str)

# on garde les arrêts dont le didok est celui d'une de nos 5 gares
arrets = stops[stops["didok"].isin(cinq["didok"])]
arrets = arrets[["stop_id", "stop_name", "platform_code", "didok"]]
arrets

In [ ]:
# les arrêts de Genève seulement
arrets[arrets["didok"] == "8501008"]

In [ ]:
# stop_times.txt est énorme : on le lit par morceaux de 2 millions de lignes,
# et seulement 2 colonnes sur 7
lecteur = pd.read_csv(dossier + "stop_times.txt", usecols=["trip_id", "stop_id"], dtype=str, chunksize=2_000_000)

# une liste vide pour ranger les petits tas qu'on garde
morceaux = []

# pour chaque morceau, on garde les lignes qui passent par un de nos 128 arrêts
for morceau in lecteur:
    morceaux.append(morceau[morceau["stop_id"].isin(arrets["stop_id"])])

# on colle tous les petits tas en un seul tableau
passages = pd.concat(morceaux)
passages

In [ ]:
# quel est le numéro (agency_id) de la compagnie RBS ?
agences = pd.read_csv(dossier + "agency.txt", dtype=str)
agences[agences["agency_name"].str.contains("Bern-Solothurn")][["agency_id", "agency_name"]]

In [ ]:
# on lit la liste des lignes de transport (routes) : chacune a un type (train, bus, tram...)
routes = pd.read_csv(dossier + "routes.txt", dtype=str)

# deux questions, chacune donne une colonne de True/False
# est-ce un train ? (types 100 à 117)
est_train = routes["route_type"].astype(int).between(100, 117)
# est-ce que ce n'est PAS le RBS (88) ? les CFF ne comptent pas ses voyageurs à Bern ("Sans RBS")
pas_rbs = routes["agency_id"] != "88"

# on garde le route_id des lignes qui répondent True aux deux
lignes_train = routes[est_train & pas_rbs]["route_id"]
len(lignes_train)

In [ ]:
# le jour qu'on choisit : mercredi 25 novembre 2026
jour = 20261125

# calendar.txt : les horaires normaux de chaque service
cal = pd.read_csv(dossier + "calendar.txt", dtype=str)

# trois questions, chacune donne une colonne de True/False
mercredi = cal["wednesday"] == "1"
commence_avant = cal["start_date"].astype(int) <= jour
finit_apres = cal["end_date"].astype(int) >= jour

# on garde les services qui répondent True aux trois questions
actifs = cal[mercredi & commence_avant & finit_apres]["service_id"]
len(actifs)

In [ ]:
# calendar_dates.txt est gros : on le lit par morceaux et on garde seulement notre jour
caldat = pd.read_csv(dossier + "calendar_dates.txt", dtype=str, chunksize=2_000_000)
# (= [] vide la boîte morceaux avant de la réutiliser)
morceaux = []
for morceau in caldat:
    morceaux.append(morceau[morceau["date"] == str(jour)])
ce_jour = pd.concat(morceaux)

# type 1 = ajouté ce jour-là, type 2 = supprimé ce jour-là
ajoutes = ce_jour[ce_jour["exception_type"] == "1"]["service_id"]
retires = ce_jour[ce_jour["exception_type"] == "2"]["service_id"]

# on met ensemble les actifs et les ajoutés, puis on enlève les supprimés
services = pd.concat([actifs, ajoutes])
services = services[~services.isin(retires)]
len(ajoutes), len(retires), len(services)

In [ ]:
# trips.txt : pour chaque trajet, sa ligne (route_id) et son calendrier (service_id)
trips = pd.read_csv(dossier + "trips.txt", usecols=["route_id", "service_id", "trip_id"], dtype=str)

# deux questions, chacune donne une colonne de True/False
est_un_train = trips["route_id"].isin(lignes_train)
roule_ce_jour = trips["service_id"].isin(services)

# on garde les trajets qui répondent True aux deux, et on sort leur trip_id
trains_du_jour = trips[est_un_train & roule_ce_jour]["trip_id"]

# dans passages, on garde seulement les passages de ces trajets-là
passages_trains = passages[passages["trip_id"].isin(trains_du_jour)]
len(trips), len(trains_du_jour), len(passages_trains)

In [ ]:
# une liste vide pour ranger le nombre de trains de chaque gare
trains = []

# pour chaque gare (son numéro didok), on compte ses arrêts de trains le 25.11
for numero in cinq["didok"]:
    # les stop_id de cette gare (toutes ses voies)
    ids = arrets[arrets["didok"] == numero]["stop_id"]
    # combien de passages de trains tombent sur un de ces stop_id
    trains.append(passages_trains["stop_id"].isin(ids).sum())

# on ajoute la liste comme nouvelle colonne du petit tableau cinq
cinq["trains"] = trains
cinq

In [ ]:
# une feuille de dessin (largeur 8, hauteur 5)
plt.figure(figsize=(8, 5))

# un point par gare : x = arrêts de trains le 25.11, y = montées + descentes par jour
plt.scatter(cinq["trains"], cinq["DTV_TJM_TGM"], s=80, color="tab:blue")

# on écrit le nom de chaque gare juste au-dessus de son point
noms = cinq["Bahnhof_Gare_Stazione"].tolist()
x = cinq["trains"].tolist()
y = cinq["DTV_TJM_TGM"].tolist()
for k in range(len(noms)):
    plt.text(x[k], y[k] + 15000, noms[k], fontsize=9, ha="center")

# l'axe des voyageurs commence à 0 pour montrer les vraies proportions
plt.ylim(0, 500000)
plt.xlim(400, 2100)

plt.title("More trains, more passengers? (5 big Swiss stations)")
plt.xlabel("Train stops on Wednesday 25.11.2026 (timetable)")
plt.ylabel("Boardings + alightings per day (2025)")
plt.figtext(0.5, -0.02, "sources: SBB open data (PF 2025), SBB GTFS timetable 2026", ha="center", fontsize=9)
plt.show()

In [ ]:
# la corrélation : un chiffre entre -1 et +1 qui dit si trains et voyageurs montent ensemble
r = cinq["trains"].corr(cinq["DTV_TJM_TGM"])
print(f"corrélation entre trains et voyageurs : {r:.2f}")

In [ ]:
# combien de montées + descentes par arrêt de train, dans chaque gare
cinq["par_train"] = (cinq["DTV_TJM_TGM"] / cinq["trains"]).round()
cinq[["Bahnhof_Gare_Stazione", "trains", "DTV_TJM_TGM", "par_train"]]

### What we found
Zurich HB est une exception. Il y a énormément de montées et de descentes (449 800 par jour), beaucoup plus que dans les autres gares, peut-être parce que c'est le centre. Les autres gares, en Suisse romande et en Suisse alémanique, sont entre environ 80 000 et 180 000 montées et descentes par jour.

Zurich HB a à la fois beaucoup de trains et beaucoup de montées et descentes, les deux vont ensemble. La corrélation entre les trains et les voyageurs est de 0,99. Zurich a aussi 232 montées et descentes par arrêt de train, contre 133 à 170 ailleurs.

Lucerne a plus de trains que Lausanne et Genève, mais ses montées et descentes n'augmentent pas autant. Il ne décolle pas comme Berne, on aurait pu s'attendre à ce qu'il soit un peu plus haut.

Limits
On a comparé seulement 5 gares, on n'a pas mis toutes les gares de Suisse. Du coup, le résultat est un peu fragile.

Les montées et descentes viennent du comptage de 2025, alors que les trains sont comptés sur une date de 2026 (le mercredi 25 novembre). La comparaison est donc seulement approximative.

Pour Berne, on a enlevé les trains RBS, parce que les CFF ne comptent pas les voyageurs du RBS (« Sans RBS »). Comme ça, on compare la même chose des deux côtés.

Pour Genève, les CFF ne comptent qu'en partie les voyageurs des trains qui viennent de l'étranger ou qui y vont, donc son chiffre est peut-être un peu trop bas. Lausanne n'a pas de remarque.

Click to add a cell.